In [1]:
import pandas as pd
import numpy as np

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

# ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Visualization
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("Head Injured Players.csv")

# Preview
df.head()

,Player,2012/2013 - Number of Injuries,2012/2013 - Games Missed,2013/2014 - Number of Injuries,2013/2014 - Games Missed,2014/2015 - Number of Injuries,2014/2015 - Games Missed,Total Number of Injuries (2012-2014),Total Games Missed (2012-2014),Age first concussion (2012-2014),Team(s) during concussion incidents 2012-2014,Roles during injuries,Current Age (approx.),Date of Birth,Biography,Wikipedia Url,Image
0,DeAndre Hopkins,NaN,NaN,1.0,0.0,NaN,NaN,1,0,21.26,Houston Texans,Wide Receiver,22,6/06/1992,"DeAndre Rashaun Hopkins (born June 6, 1992), a...",http://en.wikipedia.org/wiki/DeAndre_Hopkins,//upload.wikimedia.org/wikipedia/commons/6/60/...
1,Le'Veon Bell,NaN,NaN,1.0,0.0,NaN,NaN,1,0,21.78,Pittsburgh Steelers,Running Back,23,18/02/1992,Le'Veon Bell (pronounced LAY-vee-on; born Febr...,http://en.wikipedia.org/wiki/Le'Veon_Bell,//upload.wikimedia.org/wikipedia/commons/b/b7/...
2,Tony Jefferson,NaN,NaN,NaN,NaN,1.0,0.0,1,0,22.75,Arizona Cardinals,Safety,23,27/01/1992,"Tony Jefferson (born January 27, 1992) is an A...",http://en.wikipedia.org/wiki/Tony_Jefferson,http://a.espncdn.com/combiner/i?img=/i/headsho...
3,James Wright,NaN,NaN,NaN,NaN,1.0,0.0,1,0,22.68,Cincinnati Bengals,Wide Receiver,23,31/12/1991,"James Earl Wright (born December 31, 1991) is ...",http://en.wikipedia.org/wiki/James_Wright_(wid...,http://a.espncdn.com/combiner/i?img=/i/headsho...
4,Joseph Randle,NaN,NaN,NaN,NaN,1.0,0.0,1,0,22.71,Dallas Cowboys,Running Back,23,29/12/1991,"Joseph Randle (born December 29, 1991) is an A...",http://en.wikipedia.org/wiki/Joseph_Randle,http://a.espncdn.com/combiner/i?img=/i/headsho...


In [3]:
# Drop duplicates
df = df.drop_duplicates()

# Handle missing values
df = df.fillna("")

# Check columns
print(df.columns)

Index(['Player', '2012/2013 - Number of Injuries', '2012/2013 - Games Missed',
       '2013/2014 - Number of Injuries', '2013/2014 - Games Missed',
       '2014/2015 - Number of Injuries', '2014/2015 - Games Missed',
       'Total Number of Injuries (2012-2014)',
       'Total Games Missed (2012-2014)', 'Age first concussion (2012-2014)',
       'Team(s) during concussion incidents 2012-2014',
       'Roles during injuries', 'Current Age (approx.)', 'Date of Birth',
       'Biography', 'Wikipedia Url', 'Image'],
      dtype='str')


In [6]:
print([col for col in df.columns])

['player', '2012/2013_-_number_of_injuries', '2012/2013_-_games_missed', '2013/2014_-_number_of_injuries', '2013/2014_-_games_missed', '2014/2015_-_number_of_injuries', '2014/2015_-_games_missed', 'total_number_of_injuries_(2012-2014)', 'total_games_missed_(2012-2014)', 'age_first_concussion_(2012-2014)', 'team(s)_during_concussion_incidents_2012-2014', 'roles_during_injuries', 'current_age_(approx.)', 'date_of_birth', 'biography', 'wikipedia_url', 'image']


In [8]:
print(list(df.columns))

['player', '2012/2013_-_number_of_injuries', '2012/2013_-_games_missed', '2013/2014_-_number_of_injuries', '2013/2014_-_games_missed', '2014/2015_-_number_of_injuries', '2014/2015_-_games_missed', 'total_number_of_injuries_(2012-2014)', 'total_games_missed_(2012-2014)', 'age_first_concussion_(2012-2014)', 'team(s)_during_concussion_incidents_2012-2014', 'roles_during_injuries', 'current_age_(approx.)', 'date_of_birth', 'biography', 'wikipedia_url', 'image']


In [10]:
for col in numerical_cols:
    if col not in df.columns:
        print(f"Missing: {col}")

Missing: Age
Missing: Matches_Played
Missing: Recovery_Days


In [13]:
print('X_text exists:', 'X_text' in globals())
print('X_num_scaled exists:', 'X_num_scaled' in globals())

X_text exists: False
X_num_scaled exists: False


In [15]:
import numpy as np

X = np.array([
    [1, 2],
    [1, 4],
    [1, 0],
    [10, 2],
    [10, 4],
    [10, 0]
])

In [17]:
X = df.sample(6)

In [20]:
print(X.dtypes)

player                                               str
2012/2013_-_number_of_injuries                    object
2012/2013_-_games_missed                          object
2013/2014_-_number_of_injuries                    object
2013/2014_-_games_missed                          object
2014/2015_-_number_of_injuries                    object
2014/2015_-_games_missed                          object
total_number_of_injuries_(2012-2014)               int64
total_games_missed_(2012-2014)                     int64
age_first_concussion_(2012-2014)                 float64
team(s)_during_concussion_incidents_2012-2014        str
roles_during_injuries                                str
current_age_(approx.)                              int64
date_of_birth                                        str
biography                                            str
wikipedia_url                                        str
image                                                str
dtype: object


In [22]:
print(df.columns)
print("Cluster" in df.columns)

Index(['player', '2012/2013_-_number_of_injuries', '2012/2013_-_games_missed',
       '2013/2014_-_number_of_injuries', '2013/2014_-_games_missed',
       '2014/2015_-_number_of_injuries', '2014/2015_-_games_missed',
       'total_number_of_injuries_(2012-2014)',
       'total_games_missed_(2012-2014)', 'age_first_concussion_(2012-2014)',
       'team(s)_during_concussion_incidents_2012-2014',
       'roles_during_injuries', 'current_age_(approx.)', 'date_of_birth',
       'biography', 'wikipedia_url', 'image'],
      dtype='str')
False


In [23]:
df.to_csv("Clustered_Head_Injured_Players.csv", index=False)